# PDDL Attack Path Evaluation
Evaluate generated PDDL attack paths: solvability, syntax, and semantic quality.

## Environment Setup

In [2]:
import sys
import os
sys.path.insert(0, '../../src')

from pathlib import Path
from cve2pddlap.evaluation import (
    create_ff_checker, create_enhsp_checker, PlannerResult
)
from cve2pddlap.core.data_loader import load_few_shot_pool

ff = create_ff_checker()
enhsp = create_enhsp_checker()

print(f'Metric-FF: {ff.ff_path}')
print(f'ENHSP:     {enhsp.jar_path}')
print('Evaluation tools loaded successfully')

Metric-FF: /Users/cuilin/claudework/llm-ap-generation/llm-ap-generation/tools/Metric-FF/ff
ENHSP:     /Users/cuilin/claudework/llm-ap-generation/llm-ap-generation/tools/enhsp/enhsp.jar
Evaluation tools loaded successfully


## Data

In [3]:
DATASET_PATH = '../../resources/data/CVE-PDDL-NNL-ReAP'

# Load all reference examples (CVE / AP pairs)
few_shot_pool = load_few_shot_pool(DATASET_PATH)
print(f'Reference examples: {len(few_shot_pool)}')
for ex in few_shot_pool[:5]:
    print(f'  {ex.key}')
print('  ...')

Reference examples: 55
  CVE-2022-1471 / AP1
  CVE-2022-40149 / AP1
  CVE-2022-40149 / AP2
  CVE-2022-40150 / AP1
  CVE-2022-40150 / AP2
  ...


## Solvability and Syntax Check
Validate generated PDDL with Metric-FF (plan finding) and ENHSP (strict syntax).

In [4]:
# Test solvability on all reference domain/problem pairs
import os

results = []
for ex in few_shot_pool:
    cve_dir = os.path.join(DATASET_PATH, ex.cve_id, ex.ap_id)
    domain_path = os.path.join(cve_dir, 'domain.pddl')
    problem_path = os.path.join(cve_dir, 'problem.pddl')

    if not os.path.exists(problem_path):
        print(f'  SKIP {ex.key}: no problem.pddl')
        continue

    r_ff = ff.check(domain_path, problem_path)
    r_enhsp = enhsp.check(domain_path, problem_path)

    results.append({
        'key': ex.key,
        'ff_solvable': r_ff.solvable,
        'ff_plan_length': r_ff.plan_length,
        'ff_cost': r_ff.plan_cost,
        'enhsp_syntax_ok': r_enhsp.success,
        'enhsp_error': r_enhsp.error,
    })

    status = '✓' if r_ff.solvable else '✗'
    syntax = '✓' if r_enhsp.success else '✗'
    print(f'  {ex.key:35s}  FF:{status} (len={r_ff.plan_length}, cost={r_ff.plan_cost})  ENHSP:{syntax}')

print(f'\nTotal: {len(results)} | '
      f'FF solvable: {sum(1 for r in results if r["ff_solvable"])} | '
      f'ENHSP syntax OK: {sum(1 for r in results if r["enhsp_syntax_ok"])}')

  CVE-2022-1471 / AP1                  FF:✓ (len=13, cost=15.0)  ENHSP:✓
  CVE-2022-40149 / AP1                 FF:✓ (len=8, cost=37.0)  ENHSP:✓
  CVE-2022-40149 / AP2                 FF:✓ (len=9, cost=38.0)  ENHSP:✓
  CVE-2022-40150 / AP1                 FF:✓ (len=8, cost=37.0)  ENHSP:✓
  CVE-2022-40150 / AP2                 FF:✓ (len=9, cost=38.0)  ENHSP:✓
  CVE-2023-2976 / AP1                  FF:✓ (len=7, cost=41.0)  ENHSP:✓
  CVE-2023-2976 / AP2                  FF:✓ (len=7, cost=41.0)  ENHSP:✓
  CVE-2023-2976 / AP3                  FF:✓ (len=6, cost=40.0)  ENHSP:✓
  CVE-2023-33202 / AP1                 FF:✓ (len=13, cost=73.0)  ENHSP:✓
  CVE-2023-33202 / AP2                 FF:✓ (len=9, cost=69.0)  ENHSP:✓
  CVE-2023-33202 / AP3                 FF:✓ (len=8, cost=68.0)  ENHSP:✓
  CVE-2023-34055 / AP1                 FF:✓ (len=4, cost=47.0)  ENHSP:✓
  CVE-2023-44487 / AP1                 FF:✓ (len=13, cost=42.0)  ENHSP:✓
  CVE-2023-46589 / AP1                 FF:✓ (len=27, cost=55.

## Evaluate Solvability and Syntax of Generated PDDL AP
Test a generated PDDL-AP file against the same checks.

# Test on a generated PDDL (modify path to your generated file)
GENERATED_DOMAIN = '../../experiments/CVE-2024-38816_deepseekr1ollama_1shot.pddl'  # adjust path
PROBLEM_FILE = os.path.join(DATASET_PATH, 'CVE-2024-38816', 'AP1', 'problem.pddl')

if os.path.exists(GENERATED_DOMAIN) and os.path.exists(PROBLEM_FILE):
    print('=== Metric-FF ===')
    r_ff = ff.check(GENERATED_DOMAIN, PROBLEM_FILE)
    print(f'Solvable: {r_ff.solvable}')
    print(f'Plan length: {r_ff.plan_length}')
    print(f'Plan cost: {r_ff.plan_cost}')
    if r_ff.error:
        print(f'Error: {r_ff.error}')
    if r_ff.plan:
        print('Plan:')
        for i, a in enumerate(r_ff.plan):
            print(f'  {i}: {a}')

    print('\n=== ENHSP (syntax) ===')
    r_enhsp = enhsp.check(GENERATED_DOMAIN, PROBLEM_FILE)
    print(f'Syntax OK: {r_enhsp.success}')
    if r_enhsp.error:
        print(f'Error: {r_enhsp.error}')
else:
    print(f'File not found: {GENERATED_DOMAIN} or {PROBLEM_FILE}')
    print('Run generation first, then adjust the path above.')

## Syntactic Metrics (future)

Synonym normalization, variable name normalization, classical metrics (TP/FP/FN/Precision/Recall/F1).

In [ ]:
# TODO: syntactic evaluation

## Semantic Metrics (future)

### Embedding Similarity
Cosine similarity between NL descriptions and generated PDDL (ref: Planning in the Dark).

In [ ]:
# TODO: embedding-based evaluation

### LLM-based Evaluation

#### Intrinsic
CVE description + generated PDDL → LLM judges quality (feasibility, atomicity, completeness).

Specification-Code embedding similarity

In [ ]:
# TODO: LLM-based intrinsic evaluation

#### Extrinsic
CVE description + reference PDDL + generated PDDL → LLM judges alignment.

In [ ]:
# TODO: LLM-based extrinsic evaluation

### Human Evaluation

5-point Likert scale, ≥3 experts, Krippendorff's alpha. Structured questions at action-level and path-level.

In [ ]:
# TODO: human evaluation questionnaire design